# SAMI — Notebook 1 · Perfil de usuarios y línea base de satisfacción

**EDA puro — solo "qué datos tenemos".** Distribuciones univariadas: una variable a la vez, sin cruces, sin eje temporal, sin lectura temática, sin mapas. Frío y factual.

Fuentes: log de interacciones del chatbot (respuestas) y formulario MEAL.

## Setup

In [ ]:
# Imports. Collapsed on purpose -- no analysis here.
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

In [ ]:
# Temporary neutral styling -- brand palette removed; matplotlib defaults for now.
# These names shadow the old palette so moved cells run unchanged. Re-add the
# real palette later by replacing THIS cell (nothing else references the brand).
_CYCLE = plt.rcParams["axes.prop_cycle"].by_key()["color"]   # matplotlib defaults
PRIMARY = _CYCLE[0]
BLUE_SEQ = _CYCLE
EARTH    = _CYCLE
CAT      = _CYCLE
INK = INK2 = "black"
MUTED = "gray"
GRID  = "#cccccc"
SURFACE = "white"

def cat_colors(n):
    """n distinct matplotlib default colors (categorical)."""
    return [_CYCLE[i % len(_CYCLE)] for i in range(n)]

def seq_colors(n):
    """single default color repeated (ordered magnitude -- one hue for now)."""
    return [_CYCLE[0]] * n

def pct_count_autopct(values, min_pct=3.0):
    """Pie label 'xx.x%\n(n)'; blank under min_pct. Label formatter, not color."""
    total = float(sum(values))
    def _fmt(pct):
        if pct < min_pct:
            return ""
        return f"{pct:.1f}%\n({int(round(pct/100*total))})"
    return _fmt

## 1. Data Load — Responses

In [ ]:
DATA_PATH = '../data_&_docs/MMC_bot_responses_Grupo_nuevo_1783087815.xlsx'

df = pd.read_excel(DATA_PATH, sheet_name='mmc bot - responses', header=2)
df = df.dropna(how='all').reset_index(drop=True)

# One row has no Name/Timestamp/other field except a stray "Questions per
# user" = 388 -- a spreadsheet artifact, not a real interaction. Drop it.
df = df[df['Name'].notna()].reset_index(drop=True)

df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Questions per user'] = pd.to_numeric(df['Questions per user'], errors='coerce')

# Consolidate _other columns: fall back to free-text when main option is NaN/"Otra"
df['city_display'] = df.apply(
    lambda r: r['City_other'] if r['City'] == 'Otra' else r['City'], axis=1
)
df['nationality_display'] = df.apply(
    lambda r: r['Nationality_other'] if pd.isna(r['Nationality']) else r['Nationality'], axis=1
)

# Canonicalize city: city_display is very messy -- the same city appears under
# many spellings ("Bogotá"/"Bogota"/"Bogotá D.C"/"Colombia Bogotá"), with mixed
# case ("Santa marta"), and with department suffixes ("Soacha Cundinamarca",
# "Barranquilla Atlántico"). We match on an accent/case-insensitive key and map
# every variant to one canonical name, and drop entries that are departments or
# a country rather than a city ("Colombia", "Cundinamarca", "Antioquia", "9").
import unicodedata

def _city_key(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return s.lower().strip().rstrip('.').strip()

CITY_CANON_KEYS = {
    'medellin': 'Medellín',
    'bogota': 'Bogotá', 'bogota dc': 'Bogotá', 'bogota d.c': 'Bogotá',
    'bogota d c': 'Bogotá', 'colombia bogota': 'Bogotá', 'bogota zipaquira': 'Bogotá',
    'cucuta': 'Cúcuta',
    'santa marta': 'Santa Marta',
    'soacha': 'Soacha', 'soacha cundinamarca': 'Soacha', 'soacha condinamarca': 'Soacha',
    'soacha, cundinamarca': 'Soacha',
    'cali': 'Cali',
    'necocli': 'Necoclí',
    'barranquilla': 'Barranquilla', 'barranquilla atlantico': 'Barranquilla',
    'riohacha': 'Riohacha', 'riohacha la guajira': 'Riohacha',
    'turbo': 'Turbo', 'turbo antioquia': 'Turbo',
    'ipiales': 'Ipiales', 'bucaramanga': 'Bucaramanga', 'maicao': 'Maicao',
    'cartagena': 'Cartagena', 'tumaco': 'Tumaco', 'pasto': 'Pasto',
    'pereira': 'Pereira', 'galapa': 'Galapa',
}
CITY_DROP = {'9', 'colombia', 'cundinamarca', 'antioquia'}

def _canon_city(c):
    if not isinstance(c, str):
        return c
    k = _city_key(c)
    if k in CITY_DROP:
        return np.nan
    return CITY_CANON_KEYS.get(k, c.strip())

df['city_display'] = df['city_display'].map(_canon_city)

# Canonicalize nationality: merge duplicate spellings of the same country so
# e.g. "Colombia" and "Colombiana" are ONE category, and drop non-country junk.
NATIONALITY_CANON = {
    'Colombiana': 'Colombia', 'Soy colombovenezolana': 'Venezuela',
    'Haiti': 'Haití', 'Panama': 'Panamá', 'Peru': 'Perú', 'Mexico': 'México',
}
INVALID_NATIONALITIES = {'1', '3', '4', 'Valyria'}
_nat = df['nationality_display'].astype('string').str.strip().replace(NATIONALITY_CANON)
df['nationality_clean'] = _nat.where(~_nat.isin(INVALID_NATIONALITIES))

# City_duration is free-text with ~20 spelling/format variants of the same 5
# duration ranges. DURATION_MAP normalizes to 5 ranges + "No especifica".
DURATION_ORDER = [
    'Menos de 1 mes', 'Entre 1 y 3 meses', 'Entre 4 y 6 meses',
    'Entre 7 meses y 1 año', 'Más de 1 año', 'No especifica',
]
DURATION_MAP = {
    'Más de 1 año': 'Más de 1 año', 'Menos de 1 mes': 'Menos de 1 mes',
    'Entre 1 y 3 meses': 'Entre 1 y 3 meses',
    'Entre 7 meses y 1 año': 'Entre 7 meses y 1 año',
    'Entre 4 y 6 meses': 'Entre 4 y 6 meses',
    '2 años': 'Más de 1 año', '15 días': 'Menos de 1 mes', '8 años': 'Más de 1 año',
    '5 años': 'Más de 1 año', '3 años': 'Más de 1 año',
    'Ya tengo una semana': 'Menos de 1 mes', '7 años': 'Más de 1 año',
    'Vendo': 'No especifica', '8ños': 'Más de 1 año', '7 años 9 meses': 'Más de 1 año',
    '5 anos': 'Más de 1 año', '2 año': 'Más de 1 año', 'Santuario': 'No especifica',
    'Acabo de llegar': 'Menos de 1 mes', '3 meses': 'Entre 1 y 3 meses',
    'Norte de Santander': 'No especifica',
}
df['city_duration_clean'] = df['City_duration'].map(DURATION_MAP)

print(f"Shape: {df.shape}")
df.dtypes

In [ ]:
df.head(3)

## 2. Data Quality — Responses

In [ ]:
# Share of records missing, per field (only fields with any gaps), high -> low.
miss = df.isnull().mean()
miss = miss[miss > 0].sort_values()

fig, ax = plt.subplots(figsize=(9, max(3, len(miss) * 0.34)))
bars = ax.barh(miss.index, miss.values * 100, color=seq_colors(len(miss)))
ax.bar_label(bars, labels=[f"{v*100:.0f}%" for v in miss.values],
             padding=4, fontsize=8, color=INK2)
ax.set_xlim(0, 100)
ax.set_xlabel('Records missing this field (% of all records)')
ax.set_ylabel('')
ax.grid(True, axis='x')
ax.set_title('How complete is each field?')
plt.tight_layout()
plt.show()

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
print("Missingness rate:")
print(missing[missing > 0].map(lambda x: f"{x:.1%}").to_string())
print(f"\nDuplicate Name entries: {df['Name'].duplicated().sum()}")
print(f"Unique users:            {df['Name'].nunique()}")
print(f"Date range:              {df['Timestamp'].min().date()} -> {df['Timestamp'].max().date()}")

## 3. Demographics

### 3.1 Nationality

In [ ]:
# Which nationalities use the chatbot, and in what proportion? Top 5 + "Otros",
# on a donut with each slice labelled by share and (count). nationality_clean
# already merged duplicate spellings and dropped junk values (Section 1).
nat = df['nationality_clean'].value_counts()
data = nat.head(5).copy()
otros = int(nat.iloc[5:].sum())
if otros:
    data['Otros'] = otros

fig, ax = plt.subplots(figsize=(7.5, 5))
wedges, _t, _a = ax.pie(
    data.values, colors=cat_colors(len(data)),
    autopct=pct_count_autopct(data.values), pctdistance=0.78,
    startangle=90, counterclock=False,
    wedgeprops=dict(width=0.42, edgecolor=SURFACE, linewidth=2),
    textprops=dict(fontsize=8, color=INK))
ax.legend(wedges, [f"{k}  ({v}, {v/data.sum()*100:.1f}%)" for k, v in data.items()],
          title='Nationality', loc='center left', bbox_to_anchor=(1.0, 0.5),
          fontsize=9, frameon=False)
ax.set_title('Who uses Sami? Nationality of users — overwhelmingly Venezuelan')
plt.tight_layout()
plt.show()

### 3.2 Gender

In [ ]:
# How is the user base distributed across genders? Share + count per slice.
gender = df['Gender'].value_counts()
fig, ax = plt.subplots(figsize=(6.5, 4.5))
wedges, _t, _a = ax.pie(
    gender.values, colors=cat_colors(len(gender)),
    autopct=pct_count_autopct(gender.values, min_pct=1.0), pctdistance=0.72,
    startangle=90, counterclock=False,
    wedgeprops=dict(width=0.45, edgecolor=SURFACE, linewidth=2),
    textprops=dict(fontsize=9, color=INK))
ax.legend(wedges, [f"{k}  ({v}, {v/gender.sum()*100:.1f}%)" for k, v in gender.items()],
          title='Gender', loc='center left', bbox_to_anchor=(1.0, 0.5),
          fontsize=9, frameon=False)
ax.set_title('Gender distribution of users')
plt.tight_layout()
plt.show()

# What does "Otro" actually mean? Surface the free-text each respondent typed.
otro_detail = df.loc[df['Gender'] == 'Otro', 'Gender_other'].dropna()
print(f"'Otro' breakdown ({len(otro_detail)} of {len(df)} users):")
for text in otro_detail:
    print(f"  - {text!r}")

### 3.3 Age

In [ ]:
# Age profile: histogram (shape) with a smoothed density curve laid over it,
# next to a box plot (centre, spread, outliers).
age = df['Age'].dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2),
                         gridspec_kw={'width_ratios': [2, 1]})

axes[0].hist(age, bins=15, density=True, color=BLUE_SEQ[0],
             edgecolor=SURFACE, linewidth=0.8)
xs = np.linspace(age.min(), age.max(), 200)
axes[0].plot(xs, gaussian_kde(age)(xs), color=PRIMARY, linewidth=2)
axes[0].axvline(age.mean(), color=CAT[5], linestyle='--', linewidth=1.5,
                label=f'Mean {age.mean():.0f}')
axes[0].axvline(age.median(), color=INK, linestyle=':', linewidth=1.6,
                label=f'Median {age.median():.0f}')
axes[0].legend(fontsize=8, frameon=False)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Density')
axes[0].grid(True, axis='y')
axes[0].set_title('Age distribution (with smoothed density)')

axes[1].boxplot(age, vert=False, patch_artist=True, widths=0.5,
                boxprops=dict(facecolor=BLUE_SEQ[0], edgecolor=PRIMARY),
                medianprops=dict(color=INK, linewidth=2),
                whiskerprops=dict(color=INK2), capprops=dict(color=INK2),
                flierprops=dict(marker='o', markersize=4, markerfacecolor=MUTED,
                                markeredgecolor='none', alpha=0.5))
axes[1].set_yticks([])
axes[1].set_xlabel('Age (years)')
axes[1].grid(True, axis='x')
axes[1].set_title('Centre, spread & outliers')

plt.tight_layout()
plt.show()
print(age.describe().round(1).to_string())

In [ ]:
# User base by generational cohort. Bins follow the standard age brackets
# (as of 2025): Alfa 1-13, Gen Z 14-29, Millennials 30-45, Gen X 46-61,
# Baby Boomers 62-80, Silent 81+.
GEN_BINS = [0, 13, 29, 45, 61, 80, 200]
GEN_LABELS = [
    'Generación Alfa (1–13)',
    'Centennials · Gen Z (14–29)',
    'Millennials (30–45)',
    'Generación X (46–61)',
    'Baby Boomers (62–80)',
    'Generación Silenciosa (81+)',
]
gen = pd.cut(df['Age'].dropna(), bins=GEN_BINS, labels=GEN_LABELS, right=True)
gc = gen.value_counts().reindex(GEN_LABELS).fillna(0).astype(int)
total = gc.sum()

fig, ax = plt.subplots(figsize=(9, 4.2))
bars = ax.barh(gc.index, gc.values, color=seq_colors(len(gc)))
ax.bar_label(bars, labels=[f"{v}  ({v/total*100:.1f}%)" for v in gc.values],
             padding=4, fontsize=8, color=INK2)
ax.invert_yaxis()                       # youngest cohort on top
ax.set_xlim(0, gc.values.max() * 1.18)
ax.set_xlabel('Users')
ax.grid(True, axis='x')
ax.set_title('User base by generational cohort — concentrated in working age')
plt.tight_layout()
plt.show()

### 3.4 Care responsibilities

In [ ]:
# How many users report having minors (children) under their care?
minors = df['Minors'].value_counts()
labels = minors.index.astype(str)
fig, ax = plt.subplots(figsize=(6, 4.5))
wedges, _t, _a = ax.pie(
    minors.values, colors=cat_colors(len(minors)),
    autopct=pct_count_autopct(minors.values, min_pct=1.0), pctdistance=0.72,
    startangle=90, counterclock=False,
    wedgeprops=dict(width=0.45, edgecolor=SURFACE, linewidth=2),
    textprops=dict(fontsize=9, color=INK))
ax.legend(wedges, [f"{k}  ({v}, {v/minors.sum()*100:.1f}%)" for k, v in zip(labels, minors.values)],
          title='Minors under care', loc='center left', bbox_to_anchor=(1.0, 0.5),
          fontsize=9, frameon=False)
ax.set_title('Do users have minors under their care?')
plt.tight_layout()
plt.show()

## 4. Geography

### 4.1 Cities — users per city

*Only the simple user count per top city. Antiquity-by-city and the map are bivariate/spatial and live in Notebook 2.*

In [ ]:
# Which cities have the most chatbot users? Where the audience concentrates.
city = df['city_display'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(city.index[::-1], city.values[::-1], color=PRIMARY)
ax.bar_label(bars, padding=4, fontsize=8, color=INK2)
ax.set_xlabel('Users')
ax.grid(True, axis='x')
ax.set_xlim(0, city.max() * 1.12)
ax.set_title('Where are the users? Top 15 cities')
plt.tight_layout()
plt.show()

## 5. Migration routes

### 5.1 Time away from country of origin

In [ ]:
# How long has it been since users left their country of origin?
away = df['Away_duration'].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(9, max(3.5, len(away) * 0.4)))
bars = ax.barh(away.index, away.values, color=PRIMARY)
ax.bar_label(bars, padding=4, fontsize=8, color=INK2)
ax.set_xlabel('Users')
ax.grid(True, axis='x')
ax.set_xlim(0, away.max() * 1.12)
ax.set_title('How long since users left their country of origin?')
plt.tight_layout()
plt.show()

## 6. Chatbot engagement

### 6.1 Topics discussed — simple frequency

In [ ]:
# What topics are users asking about? Chat_summary is free-text/LLM-generated
# with several inconsistencies (leftover prompt text, "''" separators, hashtags,
# mixed case); TOPIC_MAP normalizes down to 7 canonical categories.
TOPIC_MAP = {
    'legal documentation': 'Legal Documentation',
    'humanitarian assistance': 'Humanitarian Assistance',
    'employment': 'Employment', 'services': 'Services', 'protection': 'Protection',
    'organization search': 'Organization Search',
    'journey information': 'Journey Information',
}
raw = df['Chat_summary'].dropna()
raw = raw[~raw.str.contains('Use exactly one of these hashtags', na=False)]
topics = (
    raw.str.replace("''", ',', regex=False).str.split(',').explode()
    .str.strip().str.lstrip('#').str.replace('_', ' ', regex=False).str.lower()
    .map(TOPIC_MAP).dropna().value_counts()
)

fig, ax = plt.subplots(figsize=(9, max(4, len(topics) * 0.45)))
bars = ax.barh(topics.index[::-1], topics.values[::-1], color=PRIMARY)
ax.bar_label(bars, padding=4, fontsize=8, color=INK2)
ax.set_xlabel('Mentions')
ax.grid(True, axis='x')
ax.set_xlim(0, topics.max() * 1.12)
ax.set_title('What do users ask Sami about? Topics discussed')
plt.tight_layout()
plt.show()

### 6.2 Questions per user

In [ ]:
# How many questions does a typical user ask? A histogram isn't useful (almost
# everyone asks 0-4), so a summary + distribution table communicates it better.
qs = df['Questions per user'].dropna()
print("Questions per user — summary statistics:")
print(qs.describe().round(2).to_string())
print()
counts = qs.value_counts().sort_index()
summary = pd.DataFrame({
    'Questions asked': counts.index.astype(int),
    'Users': counts.values,
    'Share of users (%)': (counts.values / counts.sum() * 100).round(1),
})
print("Distribution by number of questions asked:")
print(summary.to_string(index=False))

### 6.3 Survey sent

In [ ]:
# What fraction of users were sent the post-conversation survey?
survey = df['Survey sent'].fillna('Not sent').value_counts()
fig, ax = plt.subplots(figsize=(6, 4.5))
wedges, _t, _a = ax.pie(
    survey.values, colors=cat_colors(len(survey)),
    autopct=pct_count_autopct(survey.values, min_pct=1.0), pctdistance=0.72,
    startangle=90, counterclock=False,
    wedgeprops=dict(width=0.45, edgecolor=SURFACE, linewidth=2),
    textprops=dict(fontsize=9, color=INK))
ax.legend(wedges, [f"{k}  ({v}, {v/survey.sum()*100:.1f}%)" for k, v in survey.items()],
          title='Survey', loc='center left', bbox_to_anchor=(1.0, 0.5),
          fontsize=9, frameon=False)
ax.set_title('Was the follow-up survey sent?')
plt.tight_layout()
plt.show()

## 7. MEAL Feedback — Data Load & Quality

In [ ]:
# Source spreadsheets: MEAL feedback form and chatbot interaction log
MEAL_PATH = '../data_&_docs/MMC_MEAL_Group_Title_1783087939.xlsx'
RESP_PATH = '../data_&_docs/MMC_bot_responses_Grupo_nuevo_1783087815.xlsx'

# header=2: the sheet has two banner rows above the real column headers
meal = pd.read_excel(MEAL_PATH, sheet_name='mmc-meal', header=2)
meal = meal.dropna(how='all').reset_index(drop=True)  # drop blank rows left by the banner

# Replace the long Spanish question text with short, code-friendly column names
meal.columns = [
    'Name', 'Timestamp',
    'usefulness_rating', 'would_recommend',
    'recommendation_text', 'discovery_channel', 'discovery_other',
]
meal['Timestamp'] = pd.to_datetime(meal['Timestamp'], errors='coerce')

# Map the Spanish Likert labels to a 1-5 numeric scale so we can average / trend
RATING_MAP = {
    'Muy útil': 5, 'Útil': 4,
    'Medianamente útil': 3, 'Poco útil': 2, 'Nada útil': 1,
}
meal['rating_num'] = meal['usefulness_rating'].map(RATING_MAP)

print(f"MEAL records: {len(meal)}")
meal.head()

In [ ]:
# Share of records missing each field (% of all responses), high -> low.
miss = meal.isnull().mean()
miss = miss[miss > 0].sort_values()

fig, ax = plt.subplots(figsize=(8, 3.5))
if len(miss):
    bars = ax.barh(miss.index, miss.values * 100, color=seq_colors(len(miss)))
    ax.bar_label(bars, labels=[f"{v*100:.0f}%" for v in miss.values],
                 padding=4, fontsize=8, color=INK2)
else:
    ax.text(0.5, 0.5, 'No missing values', ha='center', va='center', transform=ax.transAxes)
ax.set_xlim(0, 100)
ax.set_xlabel('Records missing this field (% of all responses)')
ax.grid(True, axis='x')
ax.set_title('How complete is the MEAL feedback data?')
plt.tight_layout()
plt.show()

# How many chatbot users left MEAL feedback? (response rate)
resp_df = pd.read_excel(RESP_PATH, sheet_name='mmc bot - responses', header=2)
resp_df = resp_df.dropna(how='all').reset_index(drop=True)
total_users = resp_df['Name'].nunique()
meal_users = meal['Name'].nunique()
print(f"Total chatbot users (responses DB): {total_users}")
print(f"MEAL respondents:                   {meal_users}")
print(f"MEAL response rate:                 {meal_users / total_users * 100:.1f}%")

## 8. Satisfaction baseline

*Simple counts only — no trend, no qualitative reading. Response rate is low, so read these as indicative, not representative.*

### 8.1 Usefulness rating

In [ ]:
# Order categories least -> most useful (an ordinal scale, not by frequency).
rating_order = ['Nada útil', 'Poco útil', 'Medianamente útil', 'Útil', 'Muy útil']
rc = meal['usefulness_rating'].value_counts().reindex(rating_order, fill_value=0)
total = rc.sum()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(len(rc)), rc.values, color=seq_colors(len(rc)))
ax.set_xticks(range(len(rc)))
ax.set_xticklabels(rc.index, rotation=15, ha='right', fontsize=9)
ax.bar_label(bars, labels=[f"{v}\n{v/total*100:.0f}%" for v in rc.values],
             padding=3, fontsize=8, color=INK2)
ax.set_ylabel('Responses')
ax.set_ylim(0, rc.max() * 1.2)
ax.grid(True, axis='y')
ax.set_title('How useful did users find Sami? (rating distribution)')

mean_r = meal['rating_num'].mean()
ax.text(0.98, 0.95, f'Mean: {mean_r:.2f} / 5', transform=ax.transAxes,
        ha='right', va='top', fontsize=11, color=INK,
        bbox=dict(boxstyle='round,pad=0.3', facecolor=SURFACE, edgecolor=PRIMARY))
plt.tight_layout()
plt.show()

### 8.2 Would recommend

In [ ]:
# Would users recommend Sami? Share + count per slice.
rec = meal['would_recommend'].value_counts()
fig, ax = plt.subplots(figsize=(6.5, 4.5))
wedges, _t, _a = ax.pie(
    rec.values, colors=cat_colors(len(rec)),
    autopct=pct_count_autopct(rec.values, min_pct=1.0), pctdistance=0.72,
    startangle=90, counterclock=False,
    wedgeprops=dict(width=0.45, edgecolor=SURFACE, linewidth=2),
    textprops=dict(fontsize=9, color=INK))
ax.legend(wedges, [f"{k}  ({v}, {v/rec.sum()*100:.1f}%)" for k, v in rec.items()],
          title='Would recommend', loc='center left', bbox_to_anchor=(1.0, 0.5),
          fontsize=9, frameon=False)
ax.set_title('Would users recommend Sami to someone else?')
plt.tight_layout()
plt.show()
print(rec.to_string())

### 8.3 Discovery channel

In [ ]:
# How did users discover Sami?
disc = meal['discovery_channel'].value_counts()
fig, ax = plt.subplots(figsize=(8, max(3, len(disc) * 0.5)))
bars = ax.barh(disc.index[::-1], disc.values[::-1], color=PRIMARY)
ax.bar_label(bars, padding=4, fontsize=8, color=INK2)
ax.set_xlabel('Responses')
ax.set_xlim(0, disc.max() * 1.12)
ax.grid(True, axis='x')
ax.set_title('How did users discover Sami?')
plt.tight_layout()
plt.show()

### 8.4 Length of recommendations

*How much users write — not what they say.*

In [ ]:
# Character length of each free-text recommendation, as an engagement proxy,
# with a smoothed density curve over the histogram.
rec_text = meal['recommendation_text'].dropna()
lengths = rec_text.str.len()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=8, density=True, color=BLUE_SEQ[0],
        edgecolor=SURFACE, linewidth=0.8)
if lengths.nunique() > 1:
    xs = np.linspace(lengths.min(), lengths.max(), 200)
    ax.plot(xs, gaussian_kde(lengths)(xs), color=PRIMARY, linewidth=2)
ax.axvline(lengths.mean(), color='#6e824a', linestyle='--', linewidth=1.5,
           label=f'Mean: {lengths.mean():.0f} chars')
ax.legend(frameon=False)
ax.set_xlabel('Character length')
ax.set_ylabel('Density')
ax.grid(True, axis='y')
ax.set_title('How much did users write? Recommendation length')
plt.tight_layout()
plt.show()